In [ ]:
%%capture
!pip install pymupdf sentence-transformers chromadb

In [ ]:
%%capture
!pip install python-dotenv

In [ ]:
import fitz  # pymupdf
import os
from pathlib import Path

# Use a relative path so the notebook is portable across machines.
# Place all source PDFs inside a 'data/' folder next to this notebook.
pdf_folder = Path("data")

if not pdf_folder.exists():
    raise FileNotFoundError(
        f"PDF folder '{pdf_folder}' not found. "
        f"Please create a 'data/' directory next to this notebook "
        f"and place all source PDFs inside it."
    )

documents = []
doc_names = []

for filename in sorted(os.listdir(pdf_folder)):   # sorted Ensure that the results are reproducible
    if filename.endswith(".pdf"):
        filepath = pdf_folder / filename
        doc = fitz.open(filepath)
        text = ""
        for page in doc:
            text += page.get_text()
        documents.append(text)
        doc_names.append(filename)
        print(f"Loaded: {filename} ({len(text)} chars)")

print(f"\nTotal documents loaded: {len(documents)}")

In [ ]:
def chunk_text(text, chunk_size=200, overlap=30):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk.strip()) > 50:  # Filter out blocks that are too short
            chunks.append(chunk)
    return chunks

all_chunks = []
for doc in documents:
    chunks = chunk_text(doc)
    all_chunks.extend(chunks)

print(f"A total of {len(all_chunks)} text chunks were sliced")

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

print("Loading embedding model (first run downloads ~80MB)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

client = chromadb.Client()

# If collection already exists from a previous run, delete it first.
# This makes the notebook safely re-runnable.
collection_name = "phishing_kb"
try:
    client.delete_collection(collection_name)
    print(f"Existing collection '{collection_name}' deleted.")
except Exception:
    pass   # Collection didn't exist, which is fine

collection = client.create_collection(collection_name)

collection.add(
    documents=all_chunks,
    ids=[f"chunk_{i}" for i in range(len(all_chunks))]
)

print(f"Knowledge base built. Indexed {len(all_chunks)} chunks.")

In [ ]:
def retrieve(query, top_k=3):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    return results['documents'][0]

# Test
query = "How should employees handle suspicious phishing emails?"
results = retrieve(query)

print(f"Search：{query}\n")
for i, chunk in enumerate(results, 1):
    print(f"--- Paragraph {i} ---")
    print(chunk[:300])  #Show only the first 300 characters
    print()

In [ ]:
%%capture
# Cell 6 -  Install Groq
!pip install groq

In [ ]:
# Cell 7 - Generation module
import os
from dotenv import load_dotenv
from groq import Groq
load_dotenv() 
# Read from the environment variable to avoid key leakage
# Before running, please set this in the terminal: export GROQ_API_KEY="your_new_key" (Mac/Linux)
# Or in Windows PowerShell: $env:GROQ_API_KEY=“your_new_key”
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found. Please set it as an environment variable "
        "before running this notebook."
    )

client_groq = Groq(api_key=GROQ_API_KEY)
def generate_guidelines(query, retrieved_chunks):

    #Limit the context length to prevent token overflow.
    context = "\n\n".join(
        [str(chunk)[:300] for chunk in retrieved_chunks[:3]]
    )

    # Prompt
    prompt = f"""
You are a cybersecurity expert assistant.

Based ONLY on the following reference documents,
generate exactly 3 concise and actionable security guidelines.

Reference Documents:
{context}

Question:
{query}

Provide exactly 3 numbered guidelines.
Be specific and practical.
"""

    #Calling the Groq API
    response = client_groq.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=300,
        temperature=0.3
    )

    # return result
    return response.choices[0].message.content


# =========================
# Testing the Complete RAG Workflow
# =========================

test_query = "How should employees handle suspicious phishing emails?"

# Step 1: Search
retrieved = retrieve(test_query)

# Step 2: Generate
output = generate_guidelines(test_query, retrieved)

# Output
print(f"Question: {test_query}\n")
print("Generated security recommendations:\n")
print(output)

In [ ]:
# Cell 8 - 10 Practice Questions
test_queries = [
    "How should employees handle suspicious phishing emails?",
    "What technical controls can prevent phishing attacks?",
    "How can organizations implement multi-factor authentication to reduce phishing risk?",
    "What are the signs that an email might be a phishing attempt?",
    "How should a company respond after a successful phishing attack?",
    "What employee training programs are effective against phishing?",
    "How can email filtering systems detect phishing emails?",
    "What is spear phishing and how does it differ from regular phishing?",
    "How should organizations handle phishing attacks targeting executives?",
    "What are best practices for reporting phishing attempts within an organization?"
]

print(f"Total {len(test_queries)} test questions are ready.")

In [ ]:
# Cell 9 - Run RAG for each question and collect the results
import time

results_data = []

for i, query in enumerate(test_queries):
    print(f"Addressing the issue {i+1}/10: {query[:50]}...")
    
    # Search
    retrieved_chunks = retrieve(query, top_k=3)
    
    # Generate
    generated_answer = generate_guidelines(query, retrieved_chunks)
    
    results_data.append({
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "generated_answer": generated_answer
    })
    
    time.sleep(1)  # Avoiding API Rate Limiting

print("\n✅All issues have been resolved！")

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Use the pre-loaded embedding model
# The model has already been defined

def compute_context_relevance(query, retrieved_chunks):

    #“”“Metric 1: Assessing the Relevance Between Paragraphs and Questions”“”

    query_emb = model.encode(query, convert_to_tensor=True)
    scores = []
    for chunk in retrieved_chunks:
        chunk_emb = model.encode(chunk[:500], convert_to_tensor=True)
        score = util.cos_sim(query_emb, chunk_emb).item()
        scores.append(score)
    return round(float(np.mean(scores)), 4)

def compute_answer_relevance(query, answer):
    #“”“Metric 2: Relevance of Generated Answers to Questions”“”
    query_emb = model.encode(query, convert_to_tensor=True)
    answer_emb = model.encode(answer[:500], convert_to_tensor=True)
    score = util.cos_sim(query_emb, answer_emb).item()
    return round(float(score), 4)

def compute_faithfulness(answer, retrieved_chunks):
    #“”“Metric 3: Does the answer accurately reflect the search results (no hallucinations)?”“”
    context = " ".join([chunk[:500] for chunk in retrieved_chunks])
    answer_emb = model.encode(answer[:500], convert_to_tensor=True)
    context_emb = model.encode(context[:1000], convert_to_tensor=True)
    score = util.cos_sim(answer_emb, context_emb).item()
    return round(float(score), 4)

# Calculate metrics for all 10 questions
print(f"{'No.':<4} {'Context Rel':>12} {'Answer Rel':>11} {'Faithfulness':>13}")
print("-" * 45)

for i, result in enumerate(results_data):
    cr = compute_context_relevance(result["query"], result["retrieved_chunks"])
    ar = compute_answer_relevance(result["query"], result["generated_answer"])
    fa = compute_faithfulness(result["generated_answer"], result["retrieved_chunks"])
    
    result["context_relevance"] = cr
    result["answer_relevance"] = ar
    result["faithfulness"] = fa
    
    print(f"Q{i+1:<3} {cr:>12.4f} {ar:>11.4f} {fa:>13.4f}")

# Print the average score
avg_cr = np.mean([r["context_relevance"] for r in results_data])
avg_ar = np.mean([r["answer_relevance"] for r in results_data])
avg_fa = np.mean([r["faithfulness"] for r in results_data])

print("-" * 45)
print(f"{'Avg':<4} {avg_cr:>12.4f} {avg_ar:>11.4f} {avg_fa:>13.4f}")

In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "Query": r["query"][:60],
    "Context Relevance": r["context_relevance"],
    "Answer Relevance": r["answer_relevance"],
    "Faithfulness": r["faithfulness"]
} for r in results_data])

# Add an average row
avg_row = pd.DataFrame([{
    "Query": "AVERAGE",
    "Context Relevance": round(avg_cr, 4),
    "Answer Relevance": round(avg_ar, 4),
    "Faithfulness": round(avg_fa, 4)
}])

df = pd.concat([df, avg_row], ignore_index=True)

output_path = Path("evaluation_results.xlsx")   # Save to the same directory as the notebook
df.to_excel(output_path, index=False)
print(f"Results saved to {output_path.resolve()}")

In [ ]:
%%capture
!pip install openpyxl

In [ ]:
# ============================================================
# Cell: Error Analysis - Deep Dive into Low-Scoring Queries
# ============================================================
# This cell exports detailed inspection data for every query so we
# can manually analyse failure modes (low context relevance,
# low answer relevance, low faithfulness).

import pandas as pd
from pathlib import Path

# Identify the 3 worst queries per metric
def get_worst(metric_name, n=3):
    sorted_results = sorted(results_data, key=lambda r: r[metric_name])
    return sorted_results[:n]

print("=" * 80)
print("WORST 3 BY CONTEXT RELEVANCE (retrieval quality)")
print("=" * 80)
for r in get_worst("context_relevance"):
    print(f"\nQuery: {r['query']}")
    print(f"  Context Relevance: {r['context_relevance']:.4f}")
    print(f"  Answer Relevance:  {r['answer_relevance']:.4f}")
    print(f"  Faithfulness:      {r['faithfulness']:.4f}")
    print(f"  Retrieved chunks (first 200 chars each):")
    for i, chunk in enumerate(r['retrieved_chunks'], 1):
        print(f"    [{i}] {chunk[:200]}...")
    print(f"  Generated answer:\n{r['generated_answer']}")
    print("-" * 80)

print("\n" + "=" * 80)
print("WORST 3 BY ANSWER RELEVANCE (does answer address the question?)")
print("=" * 80)
for r in get_worst("answer_relevance"):
    print(f"\nQuery: {r['query']}")
    print(f"  Context Relevance: {r['context_relevance']:.4f}")
    print(f"  Answer Relevance:  {r['answer_relevance']:.4f}")
    print(f"  Faithfulness:      {r['faithfulness']:.4f}")
    print(f"  Generated answer:\n{r['generated_answer']}")
    print("-" * 80)

print("\n" + "=" * 80)
print("WORST 3 BY FAITHFULNESS (any hallucination beyond retrieved context?)")
print("=" * 80)
for r in get_worst("faithfulness"):
    print(f"\nQuery: {r['query']}")
    print(f"  Context Relevance: {r['context_relevance']:.4f}")
    print(f"  Answer Relevance:  {r['answer_relevance']:.4f}")
    print(f"  Faithfulness:      {r['faithfulness']:.4f}")
    print(f"  Retrieved chunks (first 300 chars each):")
    for i, chunk in enumerate(r['retrieved_chunks'], 1):
        print(f"    [{i}] {chunk[:300]}...")
    print(f"  Generated answer:\n{r['generated_answer']}")
    print("-" * 80)

# Also save the FULL detailed inspection to Excel for thorough review
detailed_records = []
for r in results_data:
    detailed_records.append({
        "Query": r["query"],
        "Context Rel": r["context_relevance"],
        "Answer Rel":  r["answer_relevance"],
        "Faithfulness": r["faithfulness"],
        "Chunk 1": r["retrieved_chunks"][0][:500] if len(r["retrieved_chunks"]) > 0 else "",
        "Chunk 2": r["retrieved_chunks"][1][:500] if len(r["retrieved_chunks"]) > 1 else "",
        "Chunk 3": r["retrieved_chunks"][2][:500] if len(r["retrieved_chunks"]) > 2 else "",
        "Generated Answer": r["generated_answer"],
    })

detail_df = pd.DataFrame(detailed_records)
detail_path = Path("error_analysis_details.xlsx")
detail_df.to_excel(detail_path, index=False)
print(f"\n[OK] Full detailed inspection saved to {detail_path.resolve()}")

In [ ]:
# ============================================================
# Cell: Ablation Study V1 - Effect of PDF Metadata Cleaning
# ============================================================
# Hypothesis: PDF headers/footers (URLs, email addresses, topic tags)
# pollute the retrieval pool. Filtering these out should improve
# Context Relevance and Faithfulness.
#
# This cell runs a parallel pipeline with cleaned chunks and
# compares against the baseline already computed in results_data.

import re
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------
# Step 1: Text cleaning function targeting PDF artefacts
# -----------------------------------------------------------------
def clean_pdf_text(text):
    """Remove common PDF header/footer/metadata noise from CISA documents."""
    # Remove URLs and email addresses
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)
    # Remove CISA social handles (e.g. @CISAgov, @CISACyber)
    text = re.sub(r'@\w+', ' ', text)
    # Remove "RELATED TOPICS: ... /topics/..." metadata blocks
    text = re.sub(r'RELATED TOPICS:.*?(?=\n|$)', ' ', text, flags=re.IGNORECASE)
    # Remove cross-reference paths like "</topics/cyber-threats-and-response>"
    text = re.sub(r'</[^>]+>', ' ', text)
    # Remove "Return to top", "Subscribe", "FOIA Requests" boilerplate
    boilerplate = [
        'Return to top', 'Subscribe', 'FOIA Requests', 'No FEAR Act',
        'Privacy Policy', 'Office of Inspector General', 'Website Feedback',
        'An official website of', "Here's how you know",
        'Commercial Routing Assistance',
    ]
    for phrase in boilerplate:
        text = re.sub(re.escape(phrase), ' ', text, flags=re.IGNORECASE)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# -----------------------------------------------------------------
# Step 2: Build cleaned chunks
# -----------------------------------------------------------------
cleaned_documents = [clean_pdf_text(doc) for doc in documents]

all_chunks_v1 = []
for doc in cleaned_documents:
    chunks = chunk_text(doc)   # reuse the same chunking function
    all_chunks_v1.extend(chunks)

print(f"Baseline chunks: {len(all_chunks)}")
print(f"V1 (cleaned) chunks: {len(all_chunks_v1)}")

# -----------------------------------------------------------------
# Step 3: Build a separate Chroma collection for V1
# -----------------------------------------------------------------
collection_name_v1 = "phishing_kb_v1"
try:
    client.delete_collection(collection_name_v1)
except Exception:
    pass

collection_v1 = client.create_collection(collection_name_v1)
collection_v1.add(
    documents=all_chunks_v1,
    ids=[f"chunk_v1_{i}" for i in range(len(all_chunks_v1))]
)

def retrieve_v1(query, top_k=3):
    results = collection_v1.query(query_texts=[query], n_results=top_k)
    return results['documents'][0]

# -----------------------------------------------------------------
# Step 4: Run all 10 queries through the V1 pipeline
# -----------------------------------------------------------------
import time
results_v1 = []
for i, query in enumerate(test_queries):
    print(f"V1 - Processing {i+1}/10: {query[:50]}...")
    retrieved_chunks = retrieve_v1(query, top_k=3)
    generated_answer = generate_guidelines(query, retrieved_chunks)
    results_v1.append({
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "generated_answer": generated_answer,
    })
    time.sleep(1)

# -----------------------------------------------------------------
# Step 5: Evaluate V1 with the same three metrics
# -----------------------------------------------------------------
for r in results_v1:
    r["context_relevance"] = compute_context_relevance(r["query"], r["retrieved_chunks"])
    r["answer_relevance"]  = compute_answer_relevance(r["query"], r["generated_answer"])
    r["faithfulness"]      = compute_faithfulness(r["generated_answer"], r["retrieved_chunks"])

# -----------------------------------------------------------------
# Step 6: Side-by-side comparison
# -----------------------------------------------------------------
comparison_rows = []
for base, v1 in zip(results_data, results_v1):
    comparison_rows.append({
        "Query": base["query"][:55],
        "CR_baseline": base["context_relevance"],
        "CR_v1": v1["context_relevance"],
        "CR_delta": round(v1["context_relevance"] - base["context_relevance"], 4),
        "AR_baseline": base["answer_relevance"],
        "AR_v1": v1["answer_relevance"],
        "AR_delta": round(v1["answer_relevance"] - base["answer_relevance"], 4),
        "F_baseline": base["faithfulness"],
        "F_v1": v1["faithfulness"],
        "F_delta": round(v1["faithfulness"] - base["faithfulness"], 4),
    })

avg_row = {
    "Query": "AVERAGE",
    "CR_baseline": round(np.mean([r["context_relevance"] for r in results_data]), 4),
    "CR_v1":       round(np.mean([r["context_relevance"] for r in results_v1]),   4),
    "CR_delta":    round(np.mean([v1["context_relevance"] - b["context_relevance"]
                                   for b, v1 in zip(results_data, results_v1)]), 4),
    "AR_baseline": round(np.mean([r["answer_relevance"] for r in results_data]), 4),
    "AR_v1":       round(np.mean([r["answer_relevance"] for r in results_v1]),   4),
    "AR_delta":    round(np.mean([v1["answer_relevance"] - b["answer_relevance"]
                                   for b, v1 in zip(results_data, results_v1)]), 4),
    "F_baseline":  round(np.mean([r["faithfulness"] for r in results_data]), 4),
    "F_v1":        round(np.mean([r["faithfulness"] for r in results_v1]),   4),
    "F_delta":     round(np.mean([v1["faithfulness"] - b["faithfulness"]
                                   for b, v1 in zip(results_data, results_v1)]), 4),
}
comparison_rows.append(avg_row)

comp_df = pd.DataFrame(comparison_rows)
print("\n" + "=" * 80)
print("ABLATION STUDY V1: BASELINE vs CLEANED-CHUNKS")
print("=" * 80)
print(comp_df.to_string(index=False))

# Save for the report
comp_path = Path("ablation_v1_comparison.xlsx")
comp_df.to_excel(comp_path, index=False)
print(f"\n[OK] Saved comparison to {comp_path.resolve()}")